# 📊 ShopSense — SQL Analysis Notebook
## Phase 2: Database Setup & Analytical Queries

**Objective:** Establish MySQL database infrastructure and execute multi-stage SQL queries to extract cohort retention, funnel metrics, RFM segmentation, and profit insights from raw e-commerce data.

### Analysis Pipeline
| Component | Purpose | Key Metric |
|-----------|---------|-----------|
| **Database Loading** | Import 8 CSV datasets into MySQL | 550,000+ total records |
| **Cohort Analysis** | Track customer retention by acquisition month | 0.5% month-1 retention |
| **Funnel Analysis** | Category-level performance metrics | 20 categories analyzed |
| **RFM Segmentation** | Customer value classification | 958 VIP customers |
| **Profit Queries** | Revenue & margin analysis with cost model | 62.3% avg margin |

### Key Assumptions
- **Cost Model**: Shipping 12%, Discount 8%, Operations R$5/order
- **Data Filtering**: Excluded cancelled and unavailable orders
- **Cohort Period**: Monthly cohorts based on first purchase date

In [ ]:
from sqlalchemy import create_engine, text
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ──────────────────────────────
# STEP 1: DATABASE SETUP & CONNECTION
# ──────────────────────────────
engine = create_engine(
    "mysql+mysqlconnector://",
    connect_args={
        "host": "127.0.0.1",
        "user": "root",
        "password": "Shifa@0828",  
        "database": "shopsense"
    }
)

# Verify connection
with engine.connect() as conn:
    print("✅ Connected successfully!")

# ──────────────────────────────
# STEP 2: LOAD RAW DATASETS INTO MYSQL
# ──────────────────────────────
files = {
    'orders': '../data/raw/olist_orders_dataset.csv',
    'order_items': '../data/raw/olist_order_items_dataset.csv',
    'customers': '../data/raw/olist_customers_dataset.csv',
    'products': '../data/raw/olist_products_dataset.csv',
    'sellers': '../data/raw/olist_sellers_dataset.csv',
    'payments': '../data/raw/olist_order_payments_dataset.csv',
    'reviews': '../data/raw/olist_order_reviews_dataset.csv',
    'product_category_name_translation': '../data/raw/product_category_name_translation.csv'
}

for table_name, filepath in files.items():
    df = pd.read_csv(filepath)
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f'Loaded {table_name}: {len(df):,} rows')

---

## ⚙️ Database Optimization

**Why run this?** The raw datasets do not contain performance indexes or calculated columns like `delivery_delay_days`. Running this step ensures your analytical queries run faster and have access to on-time delivery metrics.

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    print("⏳ Optimizing schema...")
    
    # 1. Add and calculate delivery_delay_days
    try:
        conn.execute(text("ALTER TABLE orders ADD COLUMN delivery_delay_days INT"))
    except Exception as e:
        print(f"Note: Column check skipped or failed ({e})")
        
    conn.execute(text("""
        UPDATE orders 
        SET delivery_delay_days = DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date)
        WHERE order_delivered_customer_date IS NOT NULL 
        AND order_estimated_delivery_date IS NOT NULL
    """))
    
    # 2. Create performance indexes
    index_queries = [
        "ALTER TABLE orders ADD INDEX idx_ord_cust (customer_id)",
        "ALTER TABLE orders ADD INDEX idx_ord_stat (order_status)",
        "ALTER TABLE order_items ADD INDEX idx_itm_ord (order_id)"
    ]
    
    for q in index_queries:
        try:
            conn.execute(text(q))
            print(f"✅ Index created: {q.split()[-1]}")
        except Exception:
            print(f"ℹ️ Index already exists or skipped: {q.split()[-1]}")
    
    conn.commit()
    print("✅ Optimization complete!")

---

## 📈 Cohort Retention Analysis

**Objective:** Identify customer loyalty patterns by tracking acquisition cohorts over time.

**Business Impact:** 
- Calculate **Customer Lifetime Value (CLV)**
- Measure effectiveness of marketing campaigns
- Identify seasonal drops in retention

In [ ]:
# ──────────────────────────────
# STEP 3: COHORT ANALYSIS
# ──────────────────────────────
query_cohort ="""
WITH customer_cohort AS(
    SELECT 
        c.customer_unique_id,
        MIN(DATE_FORMAT(o.order_purchase_timestamp, '%Y%m')) as cohort_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'unavailable')
    GROUP BY c.customer_unique_id
),
customer_orders AS(
    SELECT 
        c.customer_unique_id,
        DATE_FORMAT(o.order_purchase_timestamp, '%Y%m') AS order_month
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status NOT IN ('cancelled', 'unavailable')
),
cohort_data AS (
    SELECT
        co.customer_unique_id,
        cc.cohort_month,
        co.order_month,
        PERIOD_DIFF(co.order_month, cc.cohort_month) AS month_number
    FROM customer_orders co
    JOIN customer_cohort cc ON co.customer_unique_id = cc.customer_unique_id
)
SELECT
    cohort_month,
    COUNT(DISTINCT CASE WHEN month_number = 0 THEN customer_unique_id END) AS cohort_size,
    COUNT(DISTINCT CASE WHEN month_number = 1 THEN customer_unique_id END) AS month_1,
    COUNT(DISTINCT CASE WHEN month_number = 2 THEN customer_unique_id END) AS month_2,
    COUNT(DISTINCT CASE WHEN month_number = 3 THEN customer_unique_id END) AS month_3,
    COUNT(DISTINCT CASE WHEN month_number = 4 THEN customer_unique_id END) AS month_4,
    COUNT(DISTINCT CASE WHEN month_number = 5 THEN customer_unique_id END) AS month_5,
    COUNT(DISTINCT CASE WHEN month_number = 6 THEN customer_unique_id END) AS month_6
FROM cohort_data
GROUP BY cohort_month
ORDER BY cohort_month;
"""

df_cohort = pd.read_sql(query_cohort, engine)
df_cohort.to_csv('../data/processed/cohort_retention.csv', index=False)
print("✅ Cohort data saved to '../data/processed/cohort_retention.csv'")

# 📊 Visualization: Retention Heatmap
retention_matrix = df_cohort.set_index('cohort_month')
cohort_size = retention_matrix.iloc[:, 0]
retention = retention_matrix.divide(cohort_size, axis=0)

plt.figure(figsize=(12, 8))
sns.heatmap(retention.iloc[:, 1:], annot=True, fmt='.1%', cmap='YlGnBu')
plt.title('Customer Retention Rate by Cohort')
plt.show()

---

## 🔄 Category Funnel Analysis

**Objective:** Evaluate performance across different product categories to identify operational bottlenecks.

In [ ]:
query_funnel ="""
SELECT 
    COALESCE(t.product_category_name_english, p.product_category_name) AS category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue, 
    ROUND(AVG(oi.price), 2) AS avg_price,
    ROUND(AVG(o.delivery_delay_days), 1) AS avg_delay_days, 
    SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END) AS late_orders, 
    ROUND(100.0 * (SUM(CASE WHEN o.delivery_delay_days > 0 THEN 1 ELSE 0 END)
        /COUNT(*)), 1) AS late_rate_pct,
    ROUND(AVG(r.review_score), 1) AS avg_review_score
FROM orders o 
JOIN order_items oi ON o.order_id = oi.order_id 
JOIN products p ON oi.product_id = p.product_id 
LEFT JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status NOT IN ('cancelled', 'unavailable')
GROUP BY category
HAVING total_orders > 100
ORDER BY late_rate_pct DESC
LIMIT 20;  
"""

df_funnel = pd.read_sql(query_funnel, engine)
df_funnel.to_csv('../data/processed/category_funnel.csv', index=False)
print("✅ Funnel analysis saved to '../data/processed/category_funnel.csv'")
df_funnel.head()

---

## 👥 RFM Customer Segmentation

**Objective:** Categorize customers based on Recency, Frequency, and Monetary value for targeted marketing.

In [ ]:
query_rfm = """
WITH rfm_base AS (
    SELECT 
        c.customer_unique_id,
        DATEDIFF((SELECT MAX(order_purchase_timestamp) FROM orders), MAX(o.order_purchase_timestamp)) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        ROUND(SUM(oi.price), 2) AS monetary_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_unique_id
),
rfm_scored AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary_value ASC) AS m_score
    FROM rfm_base
)
SELECT *,
    CASE 
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
        WHEN r_score >= 4 AND f_score <= 2 THEN 'New Customers'
        WHEN r_score <= 2 AND f_score >= 4 THEN 'At Risk'
        ELSE 'Needs Attention'
    END AS segment
FROM rfm_scored;
"""

df_rfm = pd.read_sql(query_rfm, engine)
df_rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print("✅ RFM segments saved to '../data/processed/rfm_segments.csv'")
df_rfm.head()

---

## 💰 Profit Analysis

**Objective:** Analyze revenue, costs, and net profit margins across categories and states.

In [ ]:
query_profit = """
SELECT 
    DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS month,
    COALESCE(t.product_category_name_english, p.product_category_name) AS category,
    SUM(oi.price) AS total_revenue,
    ROUND(SUM(oi.price - (oi.price * 0.12 + oi.freight_value) - 
        (CASE WHEN r.review_score < 3 THEN oi.price * 0.08 ELSE 0 END) - 5.00), 2) AS total_profit
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN product_category_name_translation t ON p.product_category_name = t.product_category_name
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY month, category
ORDER BY month DESC;
"""

df_profit = pd.read_sql(query_profit, engine)
df_profit.to_csv('../data/processed/profit_analysis.csv', index=False)
print("✅ Profit analysis saved to '../data/processed/profit_analysis.csv'")

---

## 🌎 Geographic Profit Analysis

**Objective:** Identify the most profitable states and regions to optimize logistics and shipping costs.

In [8]:
query_geo_profit = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.price - (oi.price * 0.12 + oi.freight_value) - 
        (CASE WHEN r.review_score < 3 THEN oi.price * 0.08 ELSE 0 END) - 5.00), 2) AS total_profit,
    ROUND(100.0 * SUM(oi.price - (oi.price * 0.12 + oi.freight_value) - 
        (CASE WHEN r.review_score < 3 THEN oi.price * 0.08 ELSE 0 END) - 5.00) / SUM(oi.price), 1) AS profit_margin_pct
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY c.customer_state
ORDER BY total_profit DESC;
"""

df_geo_profit = pd.read_sql(query_geo_profit, engine)
df_geo_profit.to_csv('../data/processed/geo_profit.csv', index=False)
print("✅ Geographic profit data saved to '../data/processed/geo_profit.csv'")
df_geo_profit.head()

✅ Geographic profit data saved to '../data/processed/geo_profit.csv'


,customer_state,total_orders,total_revenue,total_profit,profit_margin_pct
0,SP,41125,5189711.44,3556150.92,68.5
1,RJ,12697,1819080.60,1190080.05,65.4
2,MG,11496,1579718.64,1035095.45,65.5
3,RS,5415,746506.09,481171.54,64.5
4,PR,4982,679710.81,443597.04,65.3
